# 🗄️ SQL — L'essentiel en 10 étapes (version débutant)

**But de ce notebook :** créer une **base de données**, y **ranger nos fichiers**,
poser des **questions en SQL**, puis **enregistrer** nos résultats dans la base.

Des exemples **courts**, **une idée par cellule**.

### 🤔 Les 3 mots à connaître

- **Base de données** : un fichier qui range les données dans des **tables** (comme des feuilles Excel).
- **SQL** : le langage pour **parler** à la base (créer, lire, modifier, supprimer).
- **SQLite** : une base de données **dans un simple fichier** `.db`, **déjà incluse dans Python**.

### 📂 Nos 4 fichiers → nos 4 tables

| Fichier | Table | Contenu |
|---|---|---|
| `TIE.csv` | `TIE` | les **clients** (sexe, date de naissance) |
| `TIE_ADR.csv` | `TIE_ADR` | le **nom** et la **ville** des clients |
| `CTR.csv` | `CTR` | les **contrats** (devise, solde, montant) |
| `TIE_X_CTR.csv` | `TIE_X_CTR` | **quel client** possède **quel contrat** |

En SQL, une case vide s'appelle **`NULL`**.

### 🗺️ Les 10 étapes

1. Importer
2. Créer la base et se connecter
3. Créer une table : `CREATE TABLE`
4. Remplir la table : `INSERT INTO`
5. Lire : `SELECT`
6. Filtrer : `WHERE`
7. Trier et compter : `ORDER BY`, `COUNT`
8. Regrouper : `GROUP BY`
9. Relier deux tables : `JOIN`
10. Modifier et enregistrer : `UPDATE`, `DELETE`, `CREATE TABLE AS`

💡 **Astuce :** exécutez les cellules **dans l'ordre** avec `Maj + Entrée`.

## Étape 1. Importer

- `sqlite3` : pour parler à la base de données (déjà dans Python, rien à installer)
- `pandas` : pour lire nos CSV et afficher les résultats en tableau (vu au Jour 2)

In [2]:
import sqlite3        # la base de données
import pandas as pd   # lire les CSV et afficher les résultats

## Étape 2. Créer la base et se connecter

`sqlite3.connect("banque.db")` :
- si le fichier `banque.db` **n'existe pas** → il est **créé** ;
- s'il **existe** → on **s'y connecte**.

La variable `conn` (pour *connexion*) est notre **lien** avec la base.

In [3]:
conn = sqlite3.connect("banque.db")   # crée (ou ouvre) le fichier banque.db
print("Connecté ✅")

Connecté ✅


Pour **envoyer** du SQL à la base, on utilise :

| Code | Rôle |
|---|---|
| `conn.execute("SQL")` | envoyer une commande (créer, ajouter, modifier...) |
| `conn.commit()` | **enregistrer** les changements dans le fichier |
| `pd.read_sql("SELECT ...", conn)` | poser une question et voir la réponse en tableau |

Pour écrire moins, on se crée un petit raccourci `sql(...)` :

In [4]:
def sql(requete):
    return pd.read_sql(requete, conn)   # envoie la question SQL et renvoie un tableau

sql("SELECT 1 + 1 AS resultat")         # notre première requête SQL !

,resultat
0,2


## Étape 3. Créer une table : `CREATE TABLE`

On donne le **nom** de la table, puis le **nom** et le **type** de chaque colonne.

| Type | Contenu |
|---|---|
| `INTEGER` | nombre entier |
| `REAL` | nombre à virgule |
| `TEXT` | texte (et les **dates** `AAAA-MM-JJ`) |

`PRIMARY KEY` = la colonne qui **identifie** chaque ligne (pas de doublon possible).

`DROP TABLE IF EXISTS` = supprimer la table si elle existe déjà (pour pouvoir relancer le notebook).

In [5]:
conn.execute("DROP TABLE IF EXISTS CTR")   # on repart de zéro

conn.execute("""
    CREATE TABLE CTR (
        IDT_AC       INTEGER PRIMARY KEY,
        DAT_OUV_CTR  TEXT,
        DAT_CLO_CTR  TEXT,
        COD_DEV      TEXT,
        SLD_CTR      REAL,
        MNT_INI      REAL
    )
""")
print("Table CTR créée ✅")

Table CTR créée ✅


In [6]:
# Les 3 autres tables, même principe
conn.execute("DROP TABLE IF EXISTS TIE")
conn.execute("CREATE TABLE TIE (IDT_PI INTEGER PRIMARY KEY, COD_SEX TEXT, DAT_NAI TEXT)")

conn.execute("DROP TABLE IF EXISTS TIE_ADR")
conn.execute("CREATE TABLE TIE_ADR (IDT_PI INTEGER PRIMARY KEY, NOM_TIE TEXT, PRN TEXT, NOM_VIL TEXT)")

conn.execute("DROP TABLE IF EXISTS TIE_X_CTR")
conn.execute("CREATE TABLE TIE_X_CTR (IDT_PI INTEGER, IDT_AC INTEGER)")

print("Tables TIE, TIE_ADR et TIE_X_CTR créées ✅")

Tables TIE, TIE_ADR et TIE_X_CTR créées ✅


## Étape 4. Remplir la table : `INSERT INTO`

Forme : `INSERT INTO table (colonnes) VALUES (valeurs)`

⚠️ Le texte s'écrit entre **apostrophes** : `'EUR'`.
⚠️ Sans `conn.commit()`, rien n'est enregistré dans le fichier !

In [7]:
# Ajouter UNE ligne à la main
conn.execute("""
    INSERT INTO CTR (IDT_AC, DAT_OUV_CTR, COD_DEV, SLD_CTR)
    VALUES (1, '2026-01-15', 'EUR', 500)
""")
conn.commit()      # on enregistre

sql("SELECT * FROM CTR")   # les colonnes non remplies sont vides (None = NULL)

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,1,2026-01-15,None,EUR,500.0,None


In [8]:
# On supprime cette ligne de test avant de charger les vraies données
conn.execute("DELETE FROM CTR WHERE IDT_AC = 1")
conn.commit()

Pour **200 lignes**, on ne va pas écrire 200 `INSERT` !
On lit le CSV avec Pandas, puis `to_sql` fait tous les `INSERT` pour nous.

- `usecols=[...]` → on ne garde que les colonnes de notre table
- `if_exists="append"` → on **ajoute** les lignes dans la table qu'on a créée

In [9]:
# 1) Lire le fichier (comme au Jour 2)
ctr = pd.read_csv("CTR.csv", sep=";", na_values=".",
                  usecols=["IDT_AC", "DAT_OUV_CTR", "DAT_CLO_CTR", "COD_DEV", "SLD_CTR", "MNT_INI"])

# 2) Ranger les lignes dans la table CTR
ctr.to_sql("CTR", conn, if_exists="append", index=False)
print(len(ctr), "contrats ajoutés ✅")

200 contrats ajoutés ✅


In [10]:
# Même chose pour les 3 autres fichiers
tie = pd.read_csv("TIE.csv", sep=";", na_values=".", usecols=["IDT_PI", "COD_SEX", "DAT_NAI"])
tie.to_sql("TIE", conn, if_exists="append", index=False)

adr = pd.read_csv("TIE_ADR.csv", sep=";", na_values=".", usecols=["IDT_PI", "NOM_TIE", "PRN", "NOM_VIL"])
adr.to_sql("TIE_ADR", conn, if_exists="append", index=False)

lien = pd.read_csv("TIE_X_CTR.csv", sep=";", usecols=["IDT_PI", "IDT_AC"])
lien.to_sql("TIE_X_CTR", conn, if_exists="append", index=False)

conn.commit()
print("Toutes les données sont dans la base ✅")

Toutes les données sont dans la base ✅


## Étape 5. Lire : `SELECT`

Forme : `SELECT colonnes FROM table`

| Code | Sens |
|---|---|
| `SELECT *` | toutes les colonnes |
| `SELECT col1, col2` | seulement certaines colonnes |
| `LIMIT 5` | seulement 5 lignes (comme `head()`) |
| `AS nouveau_nom` | renommer une colonne dans le résultat |

In [11]:
sql("SELECT * FROM CTR LIMIT 5")                  # toutes les colonnes, 5 lignes

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,65500004701,2024-05-29,2025-12-10,EUR,None,None
1,65500006391,2024-08-07,2025-02-05,EUR,None,None
2,65500007774,2022-01-12,None,EUR,None,None
3,65500008787,2025-01-07,None,EUR,None,None
4,65500014230,2025-01-18,2025-02-15,EUR,None,None


In [12]:
sql("SELECT IDT_AC, COD_DEV FROM CTR LIMIT 5")    # seulement 2 colonnes

,IDT_AC,COD_DEV
0,65500004701,EUR
1,65500006391,EUR
2,65500007774,EUR
3,65500008787,EUR
4,65500014230,EUR


In [13]:
sql("SELECT NOM_TIE AS nom, NOM_VIL AS ville FROM TIE_ADR LIMIT 5")   # renommer avec AS

,nom,ville
0,CEBBCACFZAAAFZTIER,BRUXELLES
1,VIRJXBF,BRUXELLES
2,JANSSENS,BRUSSEL
3,FNLLTOKQT,BRUXELLES
4,KTFSJB,BRUXELLES


## Étape 6. Filtrer : `WHERE`

Forme : `SELECT ... FROM table WHERE condition`

| Opérateur | Sens | Exemple |
|---|---|---|
| `=` | égal (⚠️ un seul `=`) | `COD_DEV = 'USD'` |
| `>` `<` | plus grand / plus petit | `SLD_CTR > 1000` |
| `AND` | ET | `... AND ...` |
| `OR` | OU | `... OR ...` |
| `IS NULL` | la case est vide | `DAT_CLO_CTR IS NULL` |

In [14]:
sql("SELECT IDT_AC, COD_DEV FROM CTR WHERE COD_DEV = 'USD'")      # les contrats en dollars

,IDT_AC,COD_DEV
0,65500017438,USD
1,65500292479,USD
2,65500451620,USD
3,65500451621,USD
4,65500477169,USD


In [15]:
sql("SELECT IDT_AC, SLD_CTR FROM CTR WHERE SLD_CTR > 10000")      # les soldes de plus de 10 000

,IDT_AC,SLD_CTR
0,65500163296,26168.60
1,65500204394,79156.24
2,65500451888,15348.70


In [16]:
sql("SELECT IDT_AC, COD_DEV, SLD_CTR FROM CTR WHERE COD_DEV = 'EUR' AND SLD_CTR > 10000")   # les 2 conditions

,IDT_AC,COD_DEV,SLD_CTR
0,65500163296,EUR,26168.60
1,65500204394,EUR,79156.24
2,65500451888,EUR,15348.70


In [17]:
sql("SELECT IDT_AC, COD_DEV FROM CTR WHERE COD_DEV = 'USD' OR COD_DEV = 'NOK'")   # l'une OU l'autre

,IDT_AC,COD_DEV
0,65500017438,USD
1,65500248622,NOK
2,65500292479,USD
3,65500451620,USD
4,65500451621,USD
5,65500477169,USD


In [18]:
sql("SELECT IDT_AC, DAT_CLO_CTR FROM CTR WHERE DAT_CLO_CTR IS NULL LIMIT 5")   # pas de date de clôture = contrat ouvert

,IDT_AC,DAT_CLO_CTR
0,65500007774,None
1,65500008787,None
2,65500017438,None
3,65500033145,None
4,65500033462,None


## Étape 7. Trier et compter

| Code | Rôle |
|---|---|
| `ORDER BY col` | trier du plus petit au plus grand |
| `ORDER BY col DESC` | trier du plus grand au plus petit |
| `COUNT(*)` | compter les lignes |
| `SUM(col)` / `AVG(col)` | somme / moyenne |
| `MIN(col)` / `MAX(col)` | plus petit / plus grand |

In [19]:
sql("SELECT IDT_AC, MNT_INI FROM CTR ORDER BY MNT_INI DESC LIMIT 3")   # les 3 plus gros montants

,IDT_AC,MNT_INI
0,65500317463,448500.0
1,65500385545,325000.0
2,65500327875,282000.0


In [20]:
sql("SELECT COUNT(*) AS nb_contrats FROM CTR")                        # combien de contrats ?

,nb_contrats
0,200


In [21]:
sql("SELECT COUNT(*) AS nb_usd FROM CTR WHERE COD_DEV = 'USD'")       # combien en dollars ?

,nb_usd
0,5


In [22]:
sql("SELECT SUM(MNT_INI) AS total, ROUND(AVG(MNT_INI), 2) AS moyenne, MAX(MNT_INI) AS maximum FROM CTR")

,total,moyenne,maximum
0,2990684.31,74767.11,448500.0


## Étape 8. Regrouper : `GROUP BY`

Pour calculer **par catégorie** (par devise, par ville...).

Forme : `SELECT categorie, COUNT(*) FROM table GROUP BY categorie`

C'est le `groupby` de Pandas.

In [23]:
sql("SELECT COD_DEV, COUNT(*) AS nb_contrats FROM CTR GROUP BY COD_DEV")   # nombre de contrats par devise

,COD_DEV,nb_contrats
0,AUD,1
1,EUR,193
2,NOK,1
3,USD,5


In [24]:
sql("SELECT COD_SEX, COUNT(*) AS nb_clients FROM TIE GROUP BY COD_SEX")    # nombre de clients par sexe

,COD_SEX,nb_clients
0,None,1
1,F,49
2,M,50


In [25]:
# Par ville, triées de la plus grande à la plus petite
sql("""
    SELECT NOM_VIL, COUNT(*) AS nb_clients
    FROM TIE_ADR
    GROUP BY NOM_VIL
    ORDER BY nb_clients DESC
    LIMIT 5
""")

,NOM_VIL,nb_clients
0,BRUXELLES,8
1,ANDERLECHT,6
2,TOURNAI,4
3,WAREGEM,3
4,SINT-AMANDSBERG,3


## Étape 9. Relier deux tables : `JOIN`

Le **nom** du client est dans `TIE_ADR`, son **sexe** dans `TIE`.
Les deux tables ont une colonne en commun : `IDT_PI`.

Forme : `FROM table1 JOIN table2 ON table1.colonne = table2.colonne`

C'est le `pd.merge` de Pandas.

In [26]:
sql("""
    SELECT TIE_ADR.NOM_TIE, TIE_ADR.NOM_VIL, TIE.COD_SEX
    FROM TIE_ADR
    JOIN TIE ON TIE_ADR.IDT_PI = TIE.IDT_PI
    LIMIT 5
""")

,NOM_TIE,NOM_VIL,COD_SEX
0,CEBBCACFZAAAFZTIER,BRUXELLES,None
1,VIRJXBF,BRUXELLES,M
2,JANSSENS,BRUSSEL,M
3,FNLLTOKQT,BRUXELLES,M
4,KTFSJB,BRUXELLES,M


In [27]:
# Combien de contrats a chaque client ? (TIE_X_CTR donne les contrats, TIE_ADR donne le nom)
sql("""
    SELECT TIE_ADR.NOM_TIE, COUNT(*) AS nb_contrats
    FROM TIE_X_CTR
    JOIN TIE_ADR ON TIE_X_CTR.IDT_PI = TIE_ADR.IDT_PI
    GROUP BY TIE_ADR.NOM_TIE
    ORDER BY nb_contrats DESC
""")

,NOM_TIE,nb_contrats
0,HISHAM,46
1,LOAN,38
2,JANSSENS,27
3,EDEN,24
4,INAYA,21
5,KYLIAN,20
6,ESTEVE,17
7,JEANNE,7


## Étape 10. Modifier et enregistrer

| Commande | Rôle |
|---|---|
| `UPDATE table SET col = valeur WHERE ...` | modifier des lignes |
| `DELETE FROM table WHERE ...` | supprimer des lignes |
| `CREATE TABLE nom AS SELECT ...` | **enregistrer un résultat** comme une nouvelle table |
| `conn.commit()` | **valider** les changements |
| `conn.close()` | fermer la base |

⚠️ **Toujours** mettre un `WHERE` avec `UPDATE` et `DELETE`, sinon **toutes les lignes** sont touchées !

In [28]:
# Ajouter un contrat
conn.execute("INSERT INTO CTR (IDT_AC, COD_DEV, SLD_CTR) VALUES (999, 'EUR', 100)")
conn.commit()
sql("SELECT * FROM CTR WHERE IDT_AC = 999")

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,999,None,None,EUR,100.0,None


In [29]:
# UPDATE : changer son solde
conn.execute("UPDATE CTR SET SLD_CTR = 250 WHERE IDT_AC = 999")
conn.commit()
sql("SELECT * FROM CTR WHERE IDT_AC = 999")

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,999,None,None,EUR,250.0,None


In [30]:
# DELETE : le supprimer
conn.execute("DELETE FROM CTR WHERE IDT_AC = 999")
conn.commit()
sql("SELECT * FROM CTR WHERE IDT_AC = 999")   # plus rien

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI


In [31]:
# CREATE TABLE AS : enregistrer le résultat d'une requête comme une NOUVELLE table
conn.execute("DROP TABLE IF EXISTS CONTRATS_USD")
conn.execute("CREATE TABLE CONTRATS_USD AS SELECT * FROM CTR WHERE COD_DEV = 'USD'")
conn.commit()

sql("SELECT * FROM CONTRATS_USD")

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,65500017438,2020-12-09,None,USD,2.82,None
1,65500292479,2023-02-06,2023-05-05,USD,NaN,None
2,65500451620,2018-02-23,2025-12-03,USD,0.00,None
3,65500451621,2018-02-23,2025-04-11,USD,0.00,None
4,65500477169,2023-04-03,2023-11-06,USD,NaN,None


In [32]:
# La liste des tables de notre base : la nouvelle table est bien là
sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,GROS_CONTRATS
1,CTR
2,TIE
3,TIE_ADR
4,TIE_X_CTR
5,CONTRATS_USD


In [33]:
conn.close()   # on ferme : tout est enregistré dans le fichier banque.db
print("Base fermée ✅")

Base fermée ✅


## 📋 Récapitulatif

| Étape | Code |
|---|---|
| Se connecter | `conn = sqlite3.connect("banque.db")` |
| Créer une table | `CREATE TABLE nom (col TYPE, ...)` |
| Ajouter | `INSERT INTO table (cols) VALUES (...)` ou `df.to_sql(...)` |
| Lire | `SELECT cols FROM table LIMIT 5` |
| Filtrer | `WHERE`, `AND`, `OR`, `IS NULL` |
| Trier | `ORDER BY col DESC` |
| Calculer | `COUNT(*)`, `SUM`, `AVG`, `MIN`, `MAX` |
| Regrouper | `GROUP BY` |
| Relier | `JOIN ... ON ...` |
| Modifier | `UPDATE ... SET ... WHERE`, `DELETE FROM ... WHERE` |
| Enregistrer | `CREATE TABLE nom AS SELECT ...`, `conn.commit()` |
| Fermer | `conn.close()` |

**SQL ↔ Pandas**

| SQL | Pandas |
|---|---|
| `WHERE` | `df[condition]` |
| `ORDER BY` | `sort_values` |
| `GROUP BY` | `groupby` |
| `JOIN` | `pd.merge` |

---

## 🎯 Mini-défis (à vous de jouer !)

**Défi 1 :** Combien de clients habitent à `BRUXELLES` ?

**Défi 2 :** Affichez les contrats en `EUR` avec un solde (`SLD_CTR`) supérieur à 20 000.

**Défi 3 :** Quel est le montant initial (`MNT_INI`) moyen par devise ?

**Défi 4 :** Enregistrez dans la base une table `GROS_CONTRATS` avec les contrats dont `MNT_INI` dépasse 100 000.

In [34]:
conn = sqlite3.connect("banque.db")   # on rouvre la base (elle a été fermée à l'étape 10)

# --- votre code ici ---

### ✅ Corrections

In [35]:
# Défi 1
sql("SELECT COUNT(*) AS nb_clients FROM TIE_ADR WHERE NOM_VIL = 'BRUXELLES'")

,nb_clients
0,8


In [36]:
# Défi 2
sql("SELECT IDT_AC, SLD_CTR FROM CTR WHERE COD_DEV = 'EUR' AND SLD_CTR > 20000")

,IDT_AC,SLD_CTR
0,65500163296,26168.60
1,65500204394,79156.24


In [37]:
# Défi 3
sql("SELECT COD_DEV, ROUND(AVG(MNT_INI), 2) AS montant_moyen FROM CTR GROUP BY COD_DEV")

,COD_DEV,montant_moyen
0,AUD,NaN
1,EUR,74767.11
2,NOK,NaN
3,USD,NaN


In [38]:
# Défi 4
conn.execute("DROP TABLE IF EXISTS GROS_CONTRATS")
conn.execute("CREATE TABLE GROS_CONTRATS AS SELECT * FROM CTR WHERE MNT_INI > 100000")
conn.commit()
sql("SELECT * FROM GROS_CONTRATS")

,IDT_AC,DAT_OUV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,MNT_INI
0,65500127184,2021-07-02,None,EUR,0.00,162500.0
1,65500136358,2022-02-11,None,EUR,-174000.00,174000.0
2,65500166207,2021-01-07,2024-10-08,EUR,0.00,181500.0
3,65500173470,2022-09-05,None,EUR,0.00,162400.0
4,65500182205,2021-07-02,None,EUR,-129247.05,162500.0
5,65500282945,2022-09-05,None,EUR,-162400.00,162400.0
6,65500296996,2022-02-11,None,EUR,0.00,174000.0
7,65500317463,2015-09-15,None,EUR,0.00,448500.0
8,65500327875,2018-07-04,None,EUR,0.00,282000.0
9,65500380735,2018-07-16,None,EUR,-201694.93,282000.0


In [39]:
conn.close()